In [1]:
import pandas as pd
import os
import sys
from omegaconf import OmegaConf
import matplotlib.pyplot as plt
import json
import joblib
import ast

import os
import json
import joblib
import inspect

import matplotlib.pyplot as plt
import numpy as np

# Adjust the path to point to external/AlphaPEM
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from src.sampling.sampler import get_polarisation_curve_samples, build_fixed_parameters
from src.validity.validity_criteria import validate_polarization_curves

### Utils

In [36]:
import numpy as np
import pandas as pd
import os
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, GroupKFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from xgboost import XGBRegressor
from itertools import product

def get_model_and_grid(model_name, random_state):
    if model_name == 'rf':
        model = RandomForestRegressor(random_state=random_state)
        param_grid = {
            'n_estimators': [100, 200, 500],
            'max_depth': [None, 5, 10],
            'min_samples_split': [2, 5, 10]
        }
    elif model_name == 'xgboost':
        model = XGBRegressor(random_state=random_state, verbosity=0)
        param_grid = {
            "n_estimators": [300, 500, 700],
            "max_depth": [6, 8, 10],
            "learning_rate": [0.01, 0.05],
            "subsample": [0.8, 1.0],
            "colsample_bytree": [0.8, 1.0]
        }
    else:
        raise ValueError(f"[ERROR] Unknown model: {model_name}")
    return model, param_grid

def compute_metrics(y_true, y_pred):
    return {
        'r2': r2_score(y_true, y_pred),
        'mse': mean_squared_error(y_true, y_pred),
        'mae': mean_absolute_error(y_true, y_pred)
    }

def nested_cv_train_with_groups(X, y, groups, feature_names, model_name='rf', outer_splits=5, inner_splits=3,
                                random_state=42, save_residuals=True):
    
    print("⚙️  Running nested CV with groups...")
    print(f"[INFO] Model: {model_name}")
    print(f"[INFO] Samples: {len(X)}, Groups: {len(np.unique(groups))}")
    print(f"[INFO] Outer Folds: {outer_splits}, Inner Folds: {inner_splits}")

    outer_cv = GroupKFold(n_splits=outer_splits)
    inner_cv = GroupKFold(n_splits=inner_splits)

    estimator, param_grid = get_model_and_grid(model_name, random_state)
    total_combinations = len(list(product(*param_grid.values())))
    print(f"[INFO] Hyperparameter grid: {total_combinations} combinations\n")

    metrics = {key: [] for key in [
        'r2', 'mse', 'mae',
        'r2_activation', 'mse_activation', 'mae_activation',
        'r2_ohmic', 'mse_ohmic', 'mae_ohmic',
        'r2_mass_transport', 'mse_mass_transport', 'mae_mass_transport'
    ]}
    best_params_list = []

    for fold, (train_idx, test_idx) in enumerate(outer_cv.split(X, y, groups=groups), 1):
        print(f"\n🔁 Fold {fold}/{outer_splits}")
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        train_groups = groups[train_idx]

        grid_search = GridSearchCV(
            estimator, param_grid,
            cv=inner_cv.split(X_train, y_train, groups=train_groups),
            scoring='r2', n_jobs=-1, verbose=0
        )
        grid_search.fit(X_train, y_train)
        best_model = grid_search.best_estimator_
        best_params_list.append(grid_search.best_params_)

        y_pred = best_model.predict(X_test)
        test_metrics = compute_metrics(y_test, y_pred)

        for key in ['r2', 'mse', 'mae']:
            metrics[key].append(test_metrics[key])

        # Region-based evaluations
        masks = {
            'activation': X_test[:, -1] < 0.4,
            'ohmic': (X_test[:, -1] >= 0.4) & (X_test[:, -1] <= 1.6),
            'mass_transport': X_test[:, -1] > 1.6
        }

        for region, mask in masks.items():
            if np.any(mask):
                region_metrics = compute_metrics(y_test[mask], best_model.predict(X_test[mask]))
            else:
                region_metrics = {'r2': np.nan, 'mse': np.nan, 'mae': np.nan}

            for key, val in region_metrics.items():
                metrics[f"{key}_{region}"].append(val)

        print(f"[INFO] Fold {fold} — R²: {test_metrics['r2']:.4f}, RMSE: {np.sqrt(test_metrics['mse']):.4f}, MAE: {test_metrics['mae']:.4f}")
    
        if save_residuals:
            os.makedirs("residuals_by_fold", exist_ok=True)

            # Convert X_test to DataFrame
            X_test_df = pd.DataFrame(X_test, columns=feature_names)

            # Create residuals DataFrame
            residuals_df = pd.DataFrame({
                'y_true': y_test,
                'y_pred': y_pred,
                'residual': y_test - y_pred
            })

            # Concatenate features
            residuals_df = pd.concat([residuals_df, X_test_df.reset_index(drop=True)], axis=1)

            # Save to CSV
            residuals_df.to_csv(f"residuals_by_fold/residuals_fold{fold}_{model_name}.csv", index=False)
            print(f"[INFO] Saved residuals and features to residuals_by_fold/residuals_fold{fold}_{model_name}.csv")

    return metrics, best_params_list

In [3]:
def single_cv_train_with_groups(X, y, groups, model_name='rf', inner_splits=3, random_state=42):
    """
    Perform a single grid search CV using GroupKFold to respect group structure.

    Returns
    -------
    best_model : trained model with best hyperparameters
    best_params : dict of best hyperparameters
    metrics : dict of evaluation metrics (R2, RMSE, MAE)
    """
    print(f"\n[INFO] Starting grid search CV for model: {model_name}")
    print(f"[INFO] Samples: {len(X)} | Unique groups: {len(np.unique(groups))} | Folds: {inner_splits}")

    inner_cv = GroupKFold(n_splits=inner_splits)
    model, param_grid = get_model_and_grid(model_name, random_state)

    grid_search = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        scoring='r2',
        cv=inner_cv.split(X, y, groups=groups),
        n_jobs=-1,
        verbose=0
    )
    grid_search.fit(X, y)

    best_model = grid_search.best_estimator_
    best_params = grid_search.best_params_

    print(f"[INFO] Best hyperparameters: {best_params}")

    y_pred = best_model.predict(X)
    r2 = r2_score(y, y_pred)
    rmse = np.sqrt(mean_squared_error(y, y_pred))
    mae = mean_absolute_error(y, y_pred)

    metrics = {
        'R2': r2,
        'RMSE': rmse,
        'MAE': mae
    }

    print(f"[INFO] Final performance on full data — R²: {r2:.4f}, RMSE: {rmse:.4f}, MAE: {mae:.4f}")

    return best_model, best_params, metrics

In [34]:
import os
import json
import inspect
import joblib
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor


def save_cv_results(
    X, y, groups,
    model_name,
    cv_fn,
    save_dir='results',
    run_name='model_run',
    inner_splits=3,
    outer_splits=5,
    random_state=42
):
    """
    Run a CV function (single or nested), train the best model on all data,
    and save metrics, hyperparameters, and the final model.

    Parameters
    ----------
    X : DataFrame or ndarray
    y : Series or ndarray
    groups : Series or ndarray
    model_name : str
        One of ["rf", "xgboost"]
    cv_fn : function
        A cross-validation function that returns either:
        - (model, best_params, metrics) for single CV
        - (metrics, best_params_list) for nested CV
    """

    os.makedirs(save_dir, exist_ok=True)
    print(f"[INFO] Running CV function: {cv_fn.__name__}")

    cv_fn_args = inspect.signature(cv_fn).parameters
    X_arr, y_arr, groups_arr = np.array(X), np.array(y), np.array(groups)

    # Decide if nested or single CV based on function signature
    if 'outer_splits' in cv_fn_args:
        # Nested CV
        metrics, best_params_list = cv_fn(
            X_arr, y_arr, groups_arr, X.columns,
            model_name=model_name,
            outer_splits=outer_splits,
            inner_splits=inner_splits,
            random_state=random_state
        )
        best_params = best_params_list[0]
    else:
        # Single CV
        model, best_params, metrics = cv_fn(
            X_arr, y_arr, groups_arr,
            model_name=model_name,
            inner_splits=inner_splits,
            random_state=random_state
        )

    # Save metrics
    metrics_path = os.path.join(save_dir, f"{run_name}_metrics.json")
    with open(metrics_path, 'w') as f:
        json.dump(metrics, f, indent=4)
    print(f"[INFO] Saved metrics to {metrics_path}")

    # Save best hyperparameters
    params_path = os.path.join(save_dir, f"{run_name}_best_params.json")
    with open(params_path, 'w') as f:
        json.dump(best_params, f, indent=4)
    print(f"[INFO] Saved best hyperparameters to {params_path}")

    # Retrain final model on all data using best parameters
    model_class = {
        'rf': RandomForestRegressor,
        'xgboost': XGBRegressor
    }.get(model_name)

    if model_class is None:
        raise ValueError(f"[ERROR] Unsupported model: {model_name}")

    final_model = model_class(random_state=random_state, **best_params)
    final_model.fit(X_arr, y_arr)

    # Save trained model
    model_path = os.path.join(save_dir, f"{run_name}_final_model.pkl")
    joblib.dump(final_model, model_path)
    print(f"[INFO] Saved trained model to {model_path}")

    return final_model, best_params, metrics


In [12]:
def ensure_numeric_dataframe(X):
    """
    Ensure that all columns in a DataFrame are numeric (float or int).
    Converts object columns to float if possible.
    Raises an error if conversion fails.

    Parameters:
    - X: pd.DataFrame

    Returns:
    - pd.DataFrame with only numeric columns
    """
    import pandas as pd

    X_checked = X.copy()
    non_numeric_cols = X_checked.select_dtypes(exclude=['number']).columns

    if len(non_numeric_cols) > 0:
        print(f"[WARN] Found non-numeric columns: {list(non_numeric_cols)}")
        for col in non_numeric_cols:
            try:
                X_checked[col] = pd.to_numeric(X_checked[col])
                print(f"[INFO] Converted column '{col}' to numeric.")
            except Exception as e:
                raise ValueError(f"[ERROR] Failed to convert column '{col}' to numeric. Reason: {e}")
    else:
        print("[INFO] All columns are already numeric.")

    return X_checked

### Training starts

In [8]:
data = pd.read_pickle(r"../sampling_test/validated_final_df_57344_imputed.pkl")
data = data[data['classification'] == 'valid']
parameter_ranges = OmegaConf.load('../param_config.yaml')
data["classification"].value_counts()

classification
valid    56312
Name: count, dtype: int64

In [9]:
criteria = {
    "start_in_range": True,
    "approx_monotonic": True,
    "low_ifc_positive_voltage": True
}

validated_df = validate_polarization_curves(
    data,
    apply_criteria=criteria,
    filter_invalid=True,
    voltage_range=(0, 1.23),
    approx_monotonic_threshold=0.05
)

print(len(validated_df))  ## Should be 54754

54754


In [15]:
columns = list(parameter_ranges.keys())
columns.extend(['Ucell', 'ifc', 'SHA256'])

n_configs = 5
sampled_df = validated_df.sample(n=n_configs, random_state=42).reset_index(drop=True)

# Explode both columns together
df_exploded = sampled_df[columns].explode(['ifc', 'Ucell'], ignore_index=True)

print("\nExploded DataFrame:")
print(len(df_exploded))
df_exploded.head()


Exploded DataFrame:
155


,Tfc,Pa_des,Sc,Phi_c_des,epsilon_gdl,tau,epsilon_mc,epsilon_c,e,Re,i0_c_ref,kappa_co,kappa_c,Ucell,ifc,SHA256
0,343.804109,214920.650007,2.431343,0.48481,0.731942,3.325909,0.289988,0.198927,5,0.000003,40.551519,33.053013,0.177528,0.971062,0.000813,f1bb7220e0a9e4d7ec051d321cbc53c20910f205e6f7fb...
1,343.804109,214920.650007,2.431343,0.48481,0.731942,3.325909,0.289988,0.198927,5,0.000003,40.551519,33.053013,0.177528,0.927253,0.100812,f1bb7220e0a9e4d7ec051d321cbc53c20910f205e6f7fb...
2,343.804109,214920.650007,2.431343,0.48481,0.731942,3.325909,0.289988,0.198927,5,0.000003,40.551519,33.053013,0.177528,0.8958,0.200812,f1bb7220e0a9e4d7ec051d321cbc53c20910f205e6f7fb...
3,343.804109,214920.650007,2.431343,0.48481,0.731942,3.325909,0.289988,0.198927,5,0.000003,40.551519,33.053013,0.177528,0.87028,0.300812,f1bb7220e0a9e4d7ec051d321cbc53c20910f205e6f7fb...
4,343.804109,214920.650007,2.431343,0.48481,0.731942,3.325909,0.289988,0.198927,5,0.000003,40.551519,33.053013,0.177528,0.848665,0.400812,f1bb7220e0a9e4d7ec051d321cbc53c20910f205e6f7fb...


In [16]:
# NEW! 
df_exploded = df_exploded[df_exploded["Ucell"] >= 0]

X = df_exploded[list(parameter_ranges.keys()) + ['ifc']]
X = ensure_numeric_dataframe(X)

y = df_exploded['Ucell'].astype(float)

[WARN] Found non-numeric columns: ['ifc']
[INFO] Converted column 'ifc' to numeric.


### XGBoost training

In [37]:
# Today, 30.07.2025
model_xgb, best_params_xgb, metrics_xgb = save_cv_results(
    X=X,
    y=y,
    groups=df_exploded['SHA256'],
    model_name='xgboost',
    cv_fn= nested_cv_train_with_groups,   # change for nested_cv_train_with_groups if outer cv is wanted
    save_dir='../results/sm/xgboost',
    run_name='test'
)

[INFO] Running CV function: nested_cv_train_with_groups
⚙️  Running nested CV with groups...
[INFO] Model: xgboost
[INFO] Samples: 155, Groups: 5
[INFO] Outer Folds: 5, Inner Folds: 3
[INFO] Hyperparameter grid: 72 combinations


🔁 Fold 1/5
[INFO] Fold 1 — R²: 0.4060, RMSE: 0.1399, MAE: 0.1391
[INFO] Saved residuals and features to residuals_by_fold/residuals_fold1_xgboost.csv

🔁 Fold 2/5
[INFO] Fold 2 — R²: 0.1634, RMSE: 0.1063, MAE: 0.0967
[INFO] Saved residuals and features to residuals_by_fold/residuals_fold2_xgboost.csv

🔁 Fold 3/5
[INFO] Fold 3 — R²: 0.9938, RMSE: 0.0119, MAE: 0.0086
[INFO] Saved residuals and features to residuals_by_fold/residuals_fold3_xgboost.csv

🔁 Fold 4/5
[INFO] Fold 4 — R²: 0.6997, RMSE: 0.0886, MAE: 0.0750
[INFO] Saved residuals and features to residuals_by_fold/residuals_fold4_xgboost.csv

🔁 Fold 5/5
[INFO] Fold 5 — R²: 0.8706, RMSE: 0.0791, MAE: 0.0624
[INFO] Saved residuals and features to residuals_by_fold/residuals_fold5_xgboost.csv
[INFO] Saved met

### Random Forest

In [12]:
columns = list(parameter_ranges.keys())
columns.extend(['Ucell', 'ifc', 'SHA256'])

n_configs = 5000
sampled_df = validated_df.sample(n=n_configs, random_state=42).reset_index(drop=True)

# Explode both columns together
df_exploded = sampled_df[columns].explode(['ifc', 'Ucell'], ignore_index=True)

print("\nExploded DataFrame:")
print(len(df_exploded))

# NEW! 
df_exploded = df_exploded[df_exploded["Ucell"] >= 0]

X = df_exploded[list(parameter_ranges.keys()) + ['ifc']]
X = ensure_numeric_dataframe(X)

y = df_exploded['Ucell'].astype(float)


Exploded DataFrame:
155000
[WARN] Found non-numeric columns: ['ifc']
[INFO] Converted column 'ifc' to numeric.


In [ ]:
# Today, 31.07.2025

model_rf, best_params_rf, metrics_rf = save_cv_results(
    X=X,
    y=y,
    groups=df_exploded['SHA256'],
    model_name='rf',
    cv_fn= nested_cv_train_with_groups,   # change for nested_cv_train_with_groups if outer cv is wanted
    save_dir='../results/sm/randomForest',
    run_name='rf_nconfig_5000_no_neg_with_outerCV_metrics_for_regions'
)

[INFO] Running CV function: nested_cv_train_with_groups
Running the new function!

[INFO] Starting nested CV for model: rf
[INFO] Total samples: 154632, grouped into 4989 unique groups
[INFO] Outer CV folds: 5, Inner CV folds: 3

[INFO] Defined hyperparameter grid with 27 combinations

[INFO] → Starting outer fold 1/5
[INFO]    Train size: (123700, 14), Test size: (30932, 14)
[INFO]    Performing inner grid search (3-fold CV)...


/opt/conda/lib/python3.10/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


## Previously

In [16]:
# Today, 29.07.2025

model_rf, best_params_rf, metrics_rf = save_cv_results(
    X=X,
    y=y,
    groups=df_exploded['SHA256'],
    model_name='rf',
    cv_fn= nested_cv_train_with_groups,   # change for nested_cv_train_with_groups if outer cv is wanted
    save_dir='../results/sm/randomForest',
    run_name='rf_nconfig_5000_no_neg_with_outerCV'
)

[INFO] Running CV function: nested_cv_train_with_groups

[INFO] Starting nested CV for model: rf
[INFO] Total samples: 154632, grouped into 4989 unique groups
[INFO] Outer CV folds: 5, Inner CV folds: 3

[INFO] Defined hyperparameter grid with 27 combinations

[INFO] → Starting outer fold 1/5
[INFO]    Train size: (123700, 14), Test size: (30932, 14)
[INFO]    Performing inner grid search (3-fold CV)...


/opt/conda/lib/python3.10/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


[INFO]    Best hyperparameters for fold 1: {'max_depth': None, 'min_samples_split': 5, 'n_estimators': 500}
[INFO]    Fold 1 performance — R²: 0.9141, RMSE: 0.0500, MAE: 0.0274

[INFO] → Starting outer fold 2/5
[INFO]    Train size: (123700, 14), Test size: (30932, 14)
[INFO]    Performing inner grid search (3-fold CV)...


/opt/conda/lib/python3.10/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


[INFO]    Best hyperparameters for fold 2: {'max_depth': None, 'min_samples_split': 5, 'n_estimators': 500}
[INFO]    Fold 2 performance — R²: 0.8865, RMSE: 0.0584, MAE: 0.0296

[INFO] → Starting outer fold 3/5
[INFO]    Train size: (123720, 14), Test size: (30912, 14)
[INFO]    Performing inner grid search (3-fold CV)...


...

[INFO]    Best hyperparameters for fold 3: {'max_depth': None, 'min_samples_split': 5, 'n_estimators': 500}
[INFO]    Fold 3 performance — R²: 0.9027, RMSE: 0.0539, MAE: 0.0287

[INFO] → Starting outer fold 4/5
[INFO]    Train size: (123701, 14), Test size: (30931, 14)
[INFO]    Performing inner grid search (3-fold CV)...


/opt/conda/lib/python3.10/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


[INFO]    Best hyperparameters for fold 4: {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 500}
[INFO]    Fold 4 performance — R²: 0.8603, RMSE: 0.0662, MAE: 0.0318

[INFO] → Starting outer fold 5/5
[INFO]    Train size: (123707, 14), Test size: (30925, 14)
[INFO]    Performing inner grid search (3-fold CV)...


/opt/conda/lib/python3.10/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


[INFO]    Best hyperparameters for fold 5: {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 500}
[INFO]    Fold 5 performance — R²: 0.8593, RMSE: 0.0658, MAE: 0.0326

[INFO] Finished all folds.
[INFO] → Summary:
   R2 mean ± std:  0.8846 ± 0.0220
   RMSE mean ± std: 0.0592 ± 0.0064
   MAE mean ± std:  0.0300 ± 0.0019

[INFO] Saved metrics to ../results/sm/randomForest/rf_nconfig_5000_no_neg_with_outerCV_metrics.json
[INFO] Saved best hyperparameters to ../results/sm/randomForest/rf_nconfig_5000_no_neg_with_outerCV_best_params.json
[INFO] Saved trained model to ../results/sm/randomForest/rf_nconfig_5000_no_neg_with_outerCV_final_model.pkl


In [24]:
model_xgb, best_params_xgb, metrics_xgb = save_cv_results(
    X=X,
    y=y,
    groups=df_exploded['SHA256'],
    model_name='xgboost',
    cv_fn=single_cv_train_with_groups,   # change for nested_cv_train_with_groups if outer cv is wanted
    save_dir='xgboost',
    run_name='xgb_nconfig10000_no_outerCV'
)

[INFO] Running CV function: single_cv_train_with_groups

[INFO] Starting single grid search for model: xgboost
[INFO] Total samples: 310000, groups: 9958
[INFO] CV folds: 3
[INFO] Performing grid search on full dataset (group-aware)...
[INFO] Best hyperparameters: {'colsample_bytree': 0.8, 'learning_rate': 0.05, 'max_depth': 8, 'n_estimators': 500, 'subsample': 1.0}
[INFO] Final model performance on full data — R²: 0.9930, RMSE: 0.0149, MAE: 0.0086

[INFO] Saved metrics to xgboost/xgb_nconfig10000_no_outerCV_metrics.json
[INFO] Saved best hyperparameters to xgboost/xgb_nconfig10000_no_outerCV_best_params.json
[INFO] Saved trained model to xgboost/xgb_nconfig10000_no_outerCV_final_model.pkl


In [29]:
model_xgb, best_params_xgb, metrics_xgb = save_cv_results(
    X=X,
    y=y,
    groups=df_exploded['SHA256'],
    model_name='xgboost',
    cv_fn= nested_cv_train_with_groups,   # change for nested_cv_train_with_groups if outer cv is wanted
    save_dir='xgboost',
    run_name='xgb_nconfig_all_with_outerCV'
)

[INFO] Running CV function: nested_cv_train_with_groups

[INFO] Starting nested CV for model: xgboost
[INFO] Total samples: 1550000, grouped into 48910 unique groups
[INFO] Outer CV folds: 5, Inner CV folds: 3

[INFO] Defined hyperparameter grid with 72 combinations

[INFO] → Starting outer fold 1/5
[INFO]    Train size: (1240000, 14), Test size: (310000, 14)
[INFO]    Performing inner grid search (3-fold CV)...
[INFO]    Best hyperparameters for fold 1: {'colsample_bytree': 0.8, 'learning_rate': 0.05, 'max_depth': 8, 'n_estimators': 700, 'subsample': 0.8}
[INFO]    Fold 1 performance — R²: 0.9329, RMSE: 0.0460, MAE: 0.0169

[INFO] → Starting outer fold 2/5
[INFO]    Train size: (1240000, 14), Test size: (310000, 14)
[INFO]    Performing inner grid search (3-fold CV)...


/opt/conda/lib/python3.10/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


[INFO]    Best hyperparameters for fold 2: {'colsample_bytree': 0.8, 'learning_rate': 0.05, 'max_depth': 8, 'n_estimators': 700, 'subsample': 0.8}
[INFO]    Fold 2 performance — R²: 0.9417, RMSE: 0.0428, MAE: 0.0162

[INFO] → Starting outer fold 3/5
[INFO]    Train size: (1240000, 14), Test size: (310000, 14)
[INFO]    Performing inner grid search (3-fold CV)...
[INFO]    Best hyperparameters for fold 3: {'colsample_bytree': 0.8, 'learning_rate': 0.05, 'max_depth': 8, 'n_estimators': 700, 'subsample': 0.8}
[INFO]    Fold 3 performance — R²: 0.9397, RMSE: 0.0435, MAE: 0.0166

[INFO] → Starting outer fold 4/5
[INFO]    Train size: (1240000, 14), Test size: (310000, 14)
[INFO]    Performing inner grid search (3-fold CV)...
[INFO]    Best hyperparameters for fold 4: {'colsample_bytree': 0.8, 'learning_rate': 0.05, 'max_depth': 8, 'n_estimators': 700, 'subsample': 1.0}
[INFO]    Fold 4 performance — R²: 0.9454, RMSE: 0.0417, MAE: 0.0161

[INFO] → Starting outer fold 5/5
[INFO]    Train size

/opt/conda/lib/python3.10/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


[INFO]    Best hyperparameters for fold 5: {'colsample_bytree': 0.8, 'learning_rate': 0.05, 'max_depth': 8, 'n_estimators': 700, 'subsample': 0.8}
[INFO]    Fold 5 performance — R²: 0.9450, RMSE: 0.0414, MAE: 0.0162

[INFO] Finished all folds.
[INFO] → Summary:
   R2 mean ± std:  0.9409 ± 0.0045
   RMSE mean ± std: 0.0431 ± 0.0017
   MAE mean ± std:  0.0164 ± 0.0003

[INFO] Saved metrics to xgboost/xgb_nconfig_all_with_outerCV_metrics.json
[INFO] Saved best hyperparameters to xgboost/xgb_nconfig_all_with_outerCV_best_params.json
[INFO] Saved trained model to xgboost/xgb_nconfig_all_with_outerCV_final_model.pkl


### Random Forest 

In [ ]:
model_rf, best_params_rf, metrics_rf = save_cv_results(
    X=X,
    y=y,
    groups=df_exploded['SHA256'],
    model_name='rf',
    cv_fn=nested_cv_train_with_groups,   # change for nested_cv_train_with_groups if outer cv is wanted
    save_dir='../results/sm/randomForest',
    run_name='rf_nconfig10000_with_outerCV'
)

[INFO] Running CV function: nested_cv_train_with_groups

[INFO] Starting nested CV for model: rf
[INFO] Total samples: 309291, grouped into 9958 unique groups
[INFO] Outer CV folds: 5, Inner CV folds: 3

[INFO] Defined hyperparameter grid with 27 combinations

[INFO] → Starting outer fold 1/5
[INFO]    Train size: (247443, 14), Test size: (61848, 14)
[INFO]    Performing inner grid search (3-fold CV)...
